# 03. 오프라인 후처리 (블록 ⑥~⑭ → 3D_final.xlsx)

| 항목 | 내용 |
|---|---|
| 입력 | `<결과 폴더>/3D_raw_online.xlsx` (02의 출력), 보정 NPZ (같은 보정값으로 광선 재계산) |
| 출력 | `<결과 폴더>/3D_final.xlsx` — Result / Legend / Plot_Aux / Summary / Configuration / Debug |
| GPU | 필요 없음 |
| 바꿀 것 | 1번 셀의 `CONFIG_PATH` 하나 (02와 같은 파일) |

블록마다 셀 하나다. 위에서부터 순서대로 실행한다. 각 셀은 블록을 실행한 뒤 간단한 집계를 출력한다.
중간 결과는 `ctx.df`(코드 내부 열 이름)에 있다. 한 번에 실행하려면 맨 아래 "한 번에 실행" 셀만 실행해도 된다.
각 단계 사이에서 불변식 검사(validate)를 수행하며, 위반하면 그 단계에서 멈춘다.

In [ ]:
#@title 0. 환경 준비 (저장소 받기 · 라이브러리 설치 · Drive 연결)
REPO_URL = "https://github.com/YOUR_ID/underwater-gripper-tracking.git"  #@param {type:"string"}

import os, sys, subprocess, warnings
REPO_DIR = '/content/' + REPO_URL.rstrip('/').split('/')[-1].removesuffix('.git')
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=False)
%pip install -q -r {REPO_DIR}/requirements.txt

from google.colab import drive
drive.mount('/content/drive')

sys.path.insert(0, REPO_DIR)
import pandas as pd
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
print('준비 완료:', REPO_DIR)

In [ ]:
#@title 1. 설정 파일 지정 · 점검
CONFIG_PATH = "/content/drive/MyDrive/8월연구/config.yaml"  #@param {type:"string"}

import shutil
from gtrack import settings

if not os.path.exists(CONFIG_PATH):
    os.makedirs(os.path.dirname(CONFIG_PATH), exist_ok=True)
    shutil.copy(os.path.join(REPO_DIR, 'config.yaml'), CONFIG_PATH)
    raise RuntimeError(
        f'설정 파일이 없어 템플릿을 복사했습니다: {CONFIG_PATH}\n'
        '왼쪽 파일 패널에서 이 파일을 더블클릭해 [필수] 항목을 수정한 뒤 이 셀을 다시 실행하세요.')

settings.configure(CONFIG_PATH)
settings.print_summary('postprocess')
settings.check('postprocess')

In [ ]:
#@title 2. 입력 읽기 (raw 엑셀 + 보정 NPZ)
from gtrack.postprocess import pipeline as P
ctx = P.step_load()
print('frames:', len(ctx.df))
print(ctx.df['Missing_Reason'].value_counts().to_string())

## ⑥ Hampel 이상치 제거

국소 선형 추세 잔차의 MAD 문턱과 축별 최소 문턱을 함께 쓴다. 이상치는 Meas를 비우고 Missing_Reason=HAMPEL_OUTLIER.  
코드: `gtrack/postprocess/b06_hampel.py`

In [ ]:
#@title ⑥ Hampel 이상치 제거
ctx = P.step06_hampel(ctx)
print('Hampel outliers:', ctx.n_outliers)

## ⑦ 복원 기준 관측 고정

Hampel 이후 실제 두 시점 좌표를 Anchor_Raw_*로 고정한다. 이후 어느 단계에서도 갱신되지 않는다.  
코드: `gtrack/postprocess/b07_anchor.py`

In [ ]:
#@title ⑦ 복원 기준 관측 고정
ctx = P.step07_anchor(ctx)
print('Anchor raw valid frames:', int(ctx.df['Anchor_Raw_Valid'].sum()))

## ⑧ 과정 잡음 σa 추정

연속 두 시점 구간 2차 차분의 강건 표준편차를 축별 σa로 쓴다.  
코드: `gtrack/postprocess/b08_process_noise.py`

In [ ]:
#@title ⑧ 과정 잡음 σa 추정
ctx = P.step08_process_noise(ctx)

## ⑨ 복원 모드 분류 · Z 하한 적용 상자 통과 판정

FRONT_MISSING → TOP_RAY, TOP_MISSING → FRONT_RAY. 상단 광선이 그리퍼 작업영역 상자를 통과하면 Top_Zone_Supported.  
코드: `gtrack/postprocess/b09_recovery_mode.py`

In [ ]:
#@title ⑨ 복원 모드 분류 · Z 하한 적용 상자 통과 판정
ctx = P.step09_recovery_mode(ctx)
print(ctx.df['Recovery_Mode'].value_counts().to_string())
print('Top_Zone_Supported frames:', int(ctx.df['Top_Zone_Supported'].sum()))

## ⑩ 단일시점 구간 좌표 복원

비관측 축(TOP_RAY→Z, FRONT_RAY→Y)을 선형 보간 / 1차원 Kalman·RTS로 추정하고, 유지된 카메라의 굴절 광선과 교차시켜 나머지 두 축을 구한다. 다이어그램 2쪽 흐름.  
코드: `gtrack/postprocess/b10_reconstruction.py, b10_hidden_axis_model.py`

In [ ]:
#@title ⑩ 단일시점 구간 좌표 복원
ctx = P.step10_reconstruction(ctx)
print(ctx.df['Filled_Type'].value_counts().to_string())
print(ctx.df['Hidden_Stop_Reason'].value_counts().to_string())

## ⑪ 작업영역 판정 · GRAB 표시

작업영역 |X|,|Y| ≤ 25, 0 ≤ Z ≤ 25 cm. 작업영역 안 기준 관측 뒤 양쪽 카메라 LOST가 60프레임 이상이면 GRAB.  
코드: `gtrack/postprocess/b11_zone_grab.py`

In [ ]:
#@title ⑪ 작업영역 판정 · GRAB 표시
ctx = P.step11_zone_grab(ctx)
print('GRAB events:', ctx.n_grab)
print(ctx.df['Coord_State'].value_counts().to_string())

## ⑫ 표시용 작업영역 연결 (Plot_Aux 전용)

그래프 음영용 Display_In_Zone. 좌표·복원·GRAB·RTS에는 반영하지 않는다.  
코드: `gtrack/postprocess/b12_display_zone.py`

In [ ]:
#@title ⑫ 표시용 작업영역 연결 (Plot_Aux 전용)
ctx = P.step12_display_zone(ctx)
print('Display-zone bridge frames:', int(ctx.df['Display_Zone_Bridge'].fillna(False).sum()))

## ⑬ 표시용 3차원 RTS 평활화 (Plot_Aux 전용)

두 시점 관측 구간만 평활화하고 복원 프레임 좌표는 보존한다. 결과는 Plot_Aux의 X_vis_cm 등.  
코드: `gtrack/postprocess/b13_rts.py`

In [ ]:
#@title ⑬ 표시용 3차원 RTS 평활화 (Plot_Aux 전용)
ctx = P.step13_rts(ctx)
print('RTS input valid frames:', int(ctx.df['RTS_Input_Valid'].sum()))

## ⑭ 프레임별 속도 산출

평활화 전 좌표의 11프레임 국소 선형 기울기. 좌표 출처 등급을 V*_Source로 기록한다.  
코드: `gtrack/postprocess/b14_velocity.py`

In [ ]:
#@title ⑭ 프레임별 속도 산출
ctx = P.step14_velocity(ctx)
print('Velocity frames:', int(ctx.df['Vx_cm_s'].notna().sum()))

## 저장: 3D_final.xlsx

In [ ]:
#@title 저장
ctx = P.step_save(ctx)

In [ ]:
#@title 결과 그래프 (Result 좌표 · 속도)
import matplotlib.pyplot as plt
res = ctx.sheets['Result']
fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
for ax, c in zip(axes, ['X_cm', 'Y_cm', 'Z_cm']):
    for state, color in (('DETECTED', 'k'), ('RECOVERED', 'tab:orange')):
        m = res['State'].eq(state)
        ax.plot(res.loc[m, 'Time_s'], res.loc[m, c], '.', ms=2, color=color, label=state)
    ax.set_ylabel(c); ax.grid(alpha=.3)
axes[0].legend(markerscale=5)
axes[3].plot(res['Time_s'], res['Speed_3D_cm_s'], lw=.8); axes[3].set_ylabel('Speed_3D [cm/s]'); axes[3].grid(alpha=.3)
axes[-1].set_xlabel('Time [s]'); plt.tight_layout(); plt.show()

## (참고) 한 번에 실행

위 셀들을 실행하지 않고 이 셀만 실행해도 같은 3D_final.xlsx가 만들어진다.

In [ ]:
#@title 한 번에 실행
from gtrack.postprocess.pipeline import run_postprocess
df_final = run_postprocess()